# uq-pet walkthrough

A thin tour of the pipeline. Every cell **imports** from `uq_pet.*` — nothing is
redefined here, so the notebook can't drift from the library. For the full run use:

```bash
uv run python -m uq_pet.main --config configs/nhr_gemma.yaml
```

In [ ]:
%load_ext autoreload
%autoreload 2

from uq_pet.config import load_config
from uq_pet.dataset import download_pet_ner, sentence_key, split_dataset, to_examples
from uq_pet.prompt import build_system_prompt, build_user_prompts, prompt_fingerprint
from uq_pet import llm, uncertainty

cfg = load_config("../configs/nhr_gemma.yaml")
cfg

## Data

Fixed split: 5 few-shot examples, a 328-sentence pool, 84 held-out test sentences.

In [ ]:
download_pet_ner()
few_shot, pool, test = split_dataset()

pool_examples = to_examples(pool)
test_examples = to_examples(test)

print(len(few_shot), len(pool_examples), len(test_examples))
pool_examples[0]

## Prompts

One constant system prefix (so it caches server-side) plus one user message per sentence.

In [ ]:
system_prompt = build_system_prompt(few_shot)
user_prompts = build_user_prompts(pool_examples)
prompt_sha = prompt_fingerprint(system_prompt)

print(prompt_sha)
print(user_prompts[0])

## Scoring

`score_split` is resumable: it reads the cache, scores only what's missing, and never
builds a client when there's nothing to do. The cell below therefore makes **no API
calls** as long as the cache is complete.

In [ ]:
records = llm.score_split(
    cfg.llm,
    system_prompt,
    user_prompts,
    [sentence_key(ex) for ex in pool_examples],
    cfg.llm.cache_path("pool"),
    prompt_sha,
)

print(f"{len(records)} records, {len(llm.failed_indices(records))} failed, "
      f"{llm.truncated_count(records)} truncated choices")
print(f"cache alignment: {llm.verify_cache_alignment(records, pool_examples):.3f}")

## Uncertainty

Scores are recomputed from the cache every time, never stored — so the variant and budget are free to change.

In [ ]:
scores = uncertainty.score_records(records, digits_only=(cfg.score == "filtered"))
ranked = uncertainty.rank_by_uncertainty(scores, tie_seed=cfg.tie_seed)

for idx in ranked[:5]:
    print(f"{scores[idx]:.3f}  {' '.join(pool_examples[idx]['tokens'])[:80]}")

## Selection

Both arms get the same number of sentences; only *which* ones differs.

In [ ]:
n = uncertainty.n_from_percent(len(pool_examples), cfg.budget_pct)
arms = {
    name: uncertainty.select(name, pool_examples, n, scores=scores,
                             seed=cfg.selection_seed, tie_seed=cfg.tie_seed)
    for name in cfg.arms
}

{name: len(examples) for name, examples in arms.items()}

## Training

Each `train_and_evaluate` call fine-tunes distilbert from scratch and scores it on the
held-out split, then frees the model. This is the slow part — the cell below runs the
full arm x seed grid, same as `main.stage_train`.

In [ ]:
from uq_pet.main import plot_arms, stage_train, summarize
from uq_pet.model_training import configure_hf_logging

configure_hf_logging(quiet=True)
results = stage_train(cfg, arms, test_examples)
results.drop(columns="per_type_f1")

In [ ]:
summary, per_type, gap = summarize(results)
display(summary)
display(per_type)
print(f"entity F1 gap (uncertainty - random): {gap:+.4f}")

In [ ]:
from pathlib import Path

out = Path("arm_f1.png")
plot_arms(results, {a: len(e) for a, e in arms.items()}, gap, out)

from IPython.display import Image
Image(str(out))